In [ ]:

# DocuMind AI - Step 1
# Check the Google Colab environment

import sys
import platform

print("Project: DocuMind AI")
print("-" * 40)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Environment setup started successfully!")


Project: DocuMind AI
----------------------------------------
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Environment setup started successfully!


In [ ]:

# DocuMind AI - Step 2
# Install the libraries required for our NLP project

%pip -q install sentence-transformers faiss-cpu pymupdf transformers sentencepiece gradio scikit-learn pandas numpy matplotlib


In [ ]:

# Verify library installation

import fitz
import faiss
import gradio as gr
import numpy as np
import pandas as pd
import sklearn
import sentence_transformers
import transformers

print("All libraries imported successfully!")
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("Gradio:", gr.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


All libraries imported successfully!
Transformers: 5.22.0
Sentence Transformers: 5.7.0
FAISS: 1.15.1
Gradio: 6.31.0
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1


In [ ]:

# DocuMind AI - Step 3.1
# Upload a research paper PDF

from google.colab import files

uploaded = files.upload()

pdf_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("Please upload at least one PDF file.")

print("\nUploaded PDF files:")
for pdf in pdf_files:
    print("-", pdf)


Saving DocuMind AI — Research Paper Intelligence Studio (1).pdf to DocuMind AI — Research Paper Intelligence Studio (1).pdf

Uploaded PDF files:
- DocuMind AI — Research Paper Intelligence Studio (1).pdf


In [ ]:

# DocuMind AI - Step 3.2
# Extract text page by page

import fitz

documents = []

for pdf_name in pdf_files:
    with fitz.open(stream=uploaded[pdf_name], filetype="pdf") as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()

            if text:
                documents.append({
                    "source": pdf_name,
                    "page": page_number,
                    "text": text
                })

print("Total non-empty pages extracted:", len(documents))

if documents:
    print("\nSource:", documents[0]["source"])
    print("Page:", documents[0]["page"])
    print("\nExtracted text preview:\n")
    print(documents[0]["text"][:1500])
else:
    print("No selectable text found. This PDF may require OCR.")


Total non-empty pages extracted: 14

Source: DocuMind AI — Research Paper Intelligence Studio (1).pdf
Page: 1

Extracted text preview:

DocuMind AI — Research Paper Intelligence Studio 
NLP Project Report · Nov 5, 2026 
Abstract 
DocuMind AI is a question-answering system for research papers: a user uploads PDFs, asks questions in 
plain English, and gets an answer together with the pages it came from. It is built with Retrieval-
Augmented Generation (RAG). PyMuPDF extracts the text, the text is cut into overlapping chunks, 
Sentence-Transformers turns each chunk into a 384-dimensional vector, FAISS finds the chunks closest 
to a question, and FLAN-T5 writes the final answer from those chunks. The project was developed in 
Google Colab in stages, from a ten-page test paper on heat-stable carbetocin in Kenya (53 chunks) to a 
multi-PDF web application with a pink/magenta Gradio interface, saved chat history, research-insight 
generation and analytics graphs. Testing showed that retrieva

In [ ]:
print("pdf_files:", "pdf_files" in globals())
print("uploaded:", "uploaded" in globals())

if "pdf_files" in globals():
    print("PDF files:", pdf_files)

pdf_files: True
uploaded: True
PDF files: ['DocuMind AI — Research Paper Intelligence Studio (1).pdf']


In [ ]:

# DocuMind AI - Step 4
# Split extracted PDF text into overlapping chunks

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = []

for doc in documents:
    text = doc["text"]
    start = 0

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "source": doc["source"],
                "page": doc["page"],
                "text": chunk_text
            })

        if end == len(text):
            break

        start = end - CHUNK_OVERLAP

print("Total extracted pages:", len(documents))
print("Total chunks created:", len(chunks))

if chunks:
    print("\nFirst chunk source:", chunks[0]["source"])
    print("First chunk page:", chunks[0]["page"])
    print("\nFirst chunk preview:\n")
    print(chunks[0]["text"][:500])
else:
    print("No chunks created. Check the PDF extraction step.")


Total extracted pages: 14
Total chunks created: 37

First chunk source: DocuMind AI — Research Paper Intelligence Studio (1).pdf
First chunk page: 1

First chunk preview:

DocuMind AI — Research Paper Intelligence Studio 
NLP Project Report · Nov 5, 2026 
Abstract 
DocuMind AI is a question-answering system for research papers: a user uploads PDFs, asks questions in 
plain English, and gets an answer together with the pages it came from. It is built with Retrieval-
Augmented Generation (RAG). PyMuPDF extracts the text, the text is cut into overlapping chunks, 
Sentence-Transformers turns each chunk into a 384-dimensional vector, FAISS finds the chunks closest 
to 


In [ ]:

# DocuMind AI - Step 5.1
# Load a pretrained Sentence Transformer model

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully!")
print("Model:", MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
Model: sentence-transformers/all-MiniLM-L6-v2


In [ ]:

# DocuMind AI - Step 5.2
# Convert all text chunks into numerical embeddings

if not chunks:
    raise ValueError("No chunks found. Please complete Step 4 first.")

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbeddings generated successfully!")
print("Total chunks:", len(chunk_texts))
print("Embeddings shape:", embeddings.shape)
print("Vector dimensions:", embeddings.shape[1])


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embeddings generated successfully!
Total chunks: 37
Embeddings shape: (37, 384)
Vector dimensions: 384


In [ ]:

# DocuMind AI - Step 6.1
# Build a FAISS index for semantic search

import faiss
import numpy as np

# Convert embeddings to the format FAISS expects
embedding_vectors = np.asarray(embeddings, dtype=np.float32)

# Create a vector index
dimension = embedding_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add all document chunk embeddings
index.add(embedding_vectors)

print("FAISS index created successfully!")
print("Total vectors indexed:", index.ntotal)
print("Vector dimensions:", dimension)


FAISS index created successfully!
Total vectors indexed: 37
Vector dimensions: 384


In [ ]:

# DocuMind AI - Step 6.2
# Search the PDF using a natural-language question

question = "What is the main objective of this research paper?"

question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True
)

question_vector = np.asarray(
    question_embedding, dtype=np.float32
)

top_k = min(3, index.ntotal)
scores, indices = index.search(question_vector, top_k)

print("Question:", question)
print("\nMost relevant passages:\n")

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]), start=1
):
    result = chunks[int(idx)]

    print(f"Result {rank} | Similarity: {score:.4f}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"][:600])
    print("-" * 60)


Question: What is the main objective of this research paper?

Most relevant passages:

Result 1 | Similarity: 0.5288
Source: DocuMind AI — Research Paper Intelligence Studio (1).pdf
Page: 4
Text: 6. Experiments and Results 
On the ten-page Kenya carbetocin paper, four test questions produced one useful answer, one 
irrelevant answer and two refusals. 
Test question 
Answer returned 
Assessment 
What is the main objective of 
this research paper? 
"I could not find this information in the 
provided documents." 
Refused, although 
page 2 states the 
study objectives 
What methodology is 
discussed in the paper? 
"To improve the quality of life of Kenyans" 
Irrelevant 
What are the main findings or 
conclusions? 
"HCPs with 11-15 years of professional 
experience had greater odds of good 
------------------------------------------------------------
Result 2 | Similarity: 0.4361
Source: DocuMind AI — Research Paper Intelligence Studio (1).pdf
Page: 4
Text: " 
Refused 
After the retrieval f

In [ ]:
doc_store = {
    "chunks": chunks,
    "index": index
}

print("Document store initialized successfully!")
print("Total chunks:", len(doc_store["chunks"]))
print("Total indexed vectors:", doc_store["index"].ntotal)

Document store initialized successfully!
Total chunks: 37
Total indexed vectors: 37


In [ ]:

# DocuMind AI - Step 7.1 FIX
# Load FLAN-T5 directly without using pipeline()

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GENERATION_MODEL = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)

model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Question-answering model loaded successfully!")
print("Model:", GENERATION_MODEL)
print("Device:", device)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question-answering model loaded successfully!
Model: google/flan-t5-base
Device: cpu


In [ ]:
print("embedding_model:", "embedding_model" in globals())
print("tokenizer:", "tokenizer" in globals())
print("model:", "model" in globals())
print("doc_store:", "doc_store" in globals())

embedding_model: True
tokenizer: True
model: True
doc_store: True


In [ ]:

# DocuMind AI - Step 7.2
# Generate answers from retrieved PDF passages

import torch
import numpy as np

def ask_documind(question, top_k=3):

    # 1. Convert question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    # 2. Retrieve relevant chunks from FAISS
    k = min(top_k, index.ntotal)
    scores, indices = index.search(question_embedding, k)

    context_parts = []
    sources = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        chunk = chunks[int(idx)]
        context_parts.append(chunk["text"])

        sources.append({
            "source": chunk["source"],
            "page": chunk["page"],
            "similarity": round(float(score), 4)
        })

    context = "\n\n".join(context_parts)

    # 3. Prepare the prompt
    prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say:
"I could not find this information in the provided documents."

Context:
{context[:2500]}

Question: {question}

Answer:
"""

    # 4. Tokenize and generate the answer
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }


# Test the system
response = ask_documind(
    "What is the main objective of this research paper?"
)

print("QUESTION:", response["question"])
print("\nAI ANSWER:", response["answer"])
print("\nSOURCES:")

for source in response["sources"]:
    print(
        f"- {source['source']} | Page {source['page']} "
        f"| Similarity: {source['similarity']}"
    )


QUESTION: What is the main objective of this research paper?

AI ANSWER: FLAN-T5-base answers are incorrect.

SOURCES:
- DocuMind AI — Research Paper Intelligence Studio (1).pdf | Page 4 | Similarity: 0.5288
- DocuMind AI — Research Paper Intelligence Studio (1).pdf | Page 4 | Similarity: 0.4361
- DocuMind AI — Research Paper Intelligence Studio (1).pdf | Page 8 | Similarity: 0.4174


In [ ]:

# DocuMind AI - Step 8.1
# Test the RAG system with multiple questions

test_questions = [
    "What is the main objective of this research paper?",
    "What methodology is discussed in the paper?",
    "What are the main findings or conclusions?",
    "What are the limitations mentioned in the paper?"
]

evaluation_results = []

for question in test_questions:
    result = ask_documind(question, top_k=3)

    evaluation_results.append({
        "Question": result["question"],
        "Generated Answer": result["answer"],
        "Source Pages": ", ".join(
            str(s["page"]) for s in result["sources"]
        )
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("DocuMind AI - Question Answering Evaluation")
display(evaluation_df)


DocuMind AI - Question Answering Evaluation


,Question,Generated Answer,Source Pages
0,What is the main objective of this research pa...,FLAN-T5-base answers are incorrect.,"4, 4, 8"
1,What methodology is discussed in the paper?,"""","4, 8, 5"
2,What are the main findings or conclusions?,is too short to be a full answer.,"4, 6, 4"
3,What are the limitations mentioned in the paper?,I could not find this information in the provi...,"4, 8, 7"


In [ ]:

# DocuMind AI - Step 9.1
# Inspect and filter low-quality retrieval results

import re
import numpy as np

def is_useful_chunk(text):
    cleaned = text.strip()
    lower = cleaned.lower()

    # Skip very short passages
    if len(cleaned) < 150:
        return False

    # Skip chunks that appear to be bibliography sections
    reference_markers = [
        "references\n",
        "bibliography\n",
        "works cited\n"
    ]

    if any(marker in lower for marker in reference_markers):
        return False

    return True


useful_indices = [
    i for i, chunk in enumerate(chunks)
    if is_useful_chunk(chunk["text"])
]

print("Original chunks:", len(chunks))
print("Useful chunks:", len(useful_indices))
print("Filtered chunks:", len(chunks) - len(useful_indices))



Original chunks: 37
Useful chunks: 37
Filtered chunks: 0


In [ ]:

# Search only among filtered chunks

filtered_embeddings = np.asarray(
    embeddings[useful_indices],
    dtype=np.float32
)

filtered_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
filtered_index.add(filtered_embeddings)

def search_documind(question, top_k=5):
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    k = min(top_k, filtered_index.ntotal)
    scores, indices = filtered_index.search(query_vector, k)

    results = []

    for score, local_idx in zip(scores[0], indices[0]):
        if local_idx < 0:
            continue

        original_idx = useful_indices[int(local_idx)]
        chunk = chunks[original_idx]

        results.append({
            "source": chunk["source"],
            "page": chunk["page"],
            "similarity": round(float(score), 4),
            "text": chunk["text"]
        })

    return results


results = search_documind(
    "What is the main objective of this research paper?"
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i} | Similarity: {result['similarity']}")
    print("Page:", result["page"])
    print("Text:", result["text"][:400])
    print("-" * 50)



Result 1 | Similarity: 0.5288
Page: 4
Text: 6. Experiments and Results 
On the ten-page Kenya carbetocin paper, four test questions produced one useful answer, one 
irrelevant answer and two refusals. 
Test question 
Answer returned 
Assessment 
What is the main objective of 
this research paper? 
"I could not find this information in the 
provided documents." 
Refused, although 
page 2 states the 
study objectives 
What methodology is 
dis
--------------------------------------------------

Result 2 | Similarity: 0.4361
Page: 4
Text: " 
Refused 
After the retrieval filter and the page-labelled prompt (Step 10), the objective question returned "HSC in 
Kenya.", which names the topic but is too short to be a full answer. 
 
similarity scores printed by the notebook (Step 6) · top 5 chunks, objective question 
Page 2 holds the study objectives but scored 0.308, just below a page of references at 0.322. This near-
tie likely contr
--------------------------------------------------

Resul

In [ ]:

# DocuMind AI - Step 10.1
# Generate answers using improved retrieval

def ask_documind_v2(question, top_k=3):

    # Retrieve relevant passages from the filtered index
    results = search_documind(question, top_k=top_k)

    if not results:
        return {
            "question": question,
            "answer": "No useful passages were found in the document.",
            "sources": []
        }

    # Build context with source page information
    context_parts = []

    for result in results:
        context_parts.append(
            f"[Source: {result['source']}, Page: {result['page']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
Answer the question using only the provided document context.
Do not invent facts.
If the context does not contain the answer, say:
"I could not find this information in the provided documents."

Context:
{context[:2200]}

Question: {question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    import torch

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "source": r["source"],
                "page": r["page"],
                "similarity": r["similarity"]
            }
            for r in results
        ]
    }


print("Improved RAG answer function is ready!")


Improved RAG answer function is ready!


In [ ]:

# Test the improved RAG pipeline

question = "What is the main objective of this research paper?"

response = ask_documind_v2(question, top_k=3)

print("QUESTION:", response["question"])
print("\nGENERATED ANSWER:\n", response["answer"])

print("\nSOURCE REFERENCES:")
for source in response["sources"]:
    print(
        f"- {source['source']} | Page {source['page']} "
        f"| Similarity: {source['similarity']}"
    )


QUESTION: What is the main objective of this research paper?

GENERATED ANSWER:
 Retrieval ranked the reference list first.

SOURCE REFERENCES:
- DocuMind AI — Research Paper Intelligence Studio (1).pdf | Page 4 | Similarity: 0.5288
- DocuMind AI — Research Paper Intelligence Studio (1).pdf | Page 4 | Similarity: 0.4361
- DocuMind AI — Research Paper Intelligence Studio (1).pdf | Page 8 | Similarity: 0.4174


In [ ]:
print("PDF chunks:", len(doc_store["chunks"]))
print("FAISS index:", doc_store["index"] is not None)

PDF chunks: 37
FAISS index: True


In [ ]:

import re
import numpy as np
import torch

def universal_qa(question, history=None):
    if not question or not question.strip():
        return "Please enter a question."

    chunks = doc_store["chunks"]
    index = doc_store["index"]

    if index is None or not chunks:
        return "Please upload and process a PDF first."

    q = question.strip()
    q_lower = q.lower()

    # Detect requests targeting particular PDF pages/slides
    page_numbers = None

    if re.search(r"\b(first|1st)\s+(two|2|2nd)\s+(slides?|pages?)\b", q_lower):
        page_numbers = {1, 2}
    else:
        match = re.search(
            r"\bpage\s+(\d+)(?:\s*(?:-|to|and)\s*(\d+))?\b",
            q_lower
        )
        if match:
            start_page = int(match.group(1))
            end_page = int(match.group(2) or start_page)
            if end_page >= start_page and end_page - start_page < 10:
                page_numbers = set(range(start_page, end_page + 1))

    # Directly select requested pages when available
    if page_numbers:
        retrieved = [
            {
                **chunk,
                "score": 1.0
            }
            for chunk in chunks
            if chunk["page"] in page_numbers
        ]

        if not retrieved:
            page_text = (
                "The requested page(s) were not found in the "
                "indexed text. Check whether the PDF's first slides "
                "are stored under different page numbers or require OCR."
            )
            return page_text

    else:
        # Semantic retrieval for general questions
        query_vector = embedding_model.encode(
            [q],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)

        k = min(6, len(chunks))
        scores, ids = index.search(query_vector, k)

        retrieved = []
        for score, idx in zip(scores[0], ids[0]):
            if idx >= 0:
                retrieved.append({
                    **chunks[int(idx)],
                    "score": float(score)
                })

    # Limit context to keep it inside the generator's input budget
    context_parts = []
    budget = 1700

    for item in retrieved:
        passage = (
            f"[PDF: {item['source']}, page {item['page']}]\n"
            f"{item['text'].strip()}"
        )

        if budget <= 0:
            break

        passage = passage[:budget]
        context_parts.append(passage)
        budget -= len(passage)

    context = "\n\n".join(context_parts)

    # Put the question first so truncation is less likely to remove it
    prompt = (
        f"Question: {q}\n"
        "Answer using only the evidence in the context. "
        "Explain clearly and include examples if requested. "
        "If evidence is insufficient, say so.\n"
        f"Context:\n{context}\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(next(model.parameters()).device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ).strip()

    sources = []
    seen_sources = set()

    for item in retrieved:
        key = (item["source"], item["page"])
        if key not in seen_sources:
            seen_sources.add(key)
            sources.append(
                f"- {item['source']}, page {item['page']}"
            )

    return (
        f"{answer or 'The model could not generate an answer.'}"
        "\n\n### Sources\n" + "\n".join(sources)
    )

print("Universal Q&A upgraded successfully!")


Universal Q&A upgraded successfully!


In [ ]:

print("process_pdfs:", "process_pdfs" in globals())
print("universal_qa:", "universal_qa" in globals())
print("embedding_model:", "embedding_model" in globals())
print("doc_store:", "doc_store" in globals())
print("tokenizer:", "tokenizer" in globals())
print("model:", "model" in globals())


process_pdfs: False
universal_qa: True
embedding_model: True
doc_store: True
tokenizer: True
model: True


In [ ]:

import gradio as gr

print("========== DOCUMIND AI STATUS ==========")

checks = {
    "PDF Processing Function": callable(globals().get("process_pdfs")),
    "Universal Q&A Function": callable(globals().get("universal_qa")),
    "Embedding Model": "embedding_model" in globals(),
    "Tokenizer": "tokenizer" in globals(),
    "Generation Model": "model" in globals(),
    "Document Store": "doc_store" in globals()
}

for name, status in checks.items():
    print(f"{name}: {'READY' if status else 'MISSING'}")

if "doc_store" in globals():
    print("Total chunks:", len(doc_store["chunks"]))
    print("FAISS index:", doc_store["index"] is not None)

print("Gradio version:", gr.__version__)
print("========================================")


========== DOCUMIND AI STATUS ==========
PDF Processing Function: MISSING
Universal Q&A Function: READY
Embedding Model: READY
Tokenizer: READY
Generation Model: READY
Document Store: READY
Total chunks: 37
FAISS index: True
Gradio version: 6.31.0


In [ ]:

import re
from collections import Counter

# Common words excluded from keyword analysis
STOP_WORDS = {
    "the", "is", "are", "was", "were", "a", "an", "and",
    "or", "of", "to", "in", "on", "for", "with", "by",
    "this", "that", "these", "those", "it", "its", "as",
    "at", "be", "been", "from", "we", "our", "their",
    "they", "he", "she", "which", "who", "what", "how",
    "can", "could", "may", "might", "will", "would",
    "using", "used", "use", "also", "such", "than",
    "into", "between", "within", "through", "based"
}


def get_document_analytics():

    chunks = doc_store["chunks"]

    if not chunks:
        return {
            "documents": 0,
            "chunks": 0,
            "pages": 0,
            "words": 0,
            "keywords": [],
            "document_list": []
        }

    sources = sorted(set(
        chunk["source"] for chunk in chunks
    ))

    pages = set(
        (chunk["source"], chunk["page"])
        for chunk in chunks
    )

    all_text = " ".join(
        chunk["text"] for chunk in chunks
    )

    words = re.findall(
        r"\b[a-zA-Z]{3,}\b",
        all_text.lower()
    )

    keywords = Counter(
        word for word in words
        if word not in STOP_WORDS
    )

    document_list = []

    for source in sources:

        source_chunks = [
            chunk for chunk in chunks
            if chunk["source"] == source
        ]

        source_pages = len(set(
            chunk["page"] for chunk in source_chunks
        ))

        document_list.append({
            "filename": source,
            "indexed_pages": source_pages,
            "chunks": len(source_chunks)
        })

    return {
        "documents": len(sources),
        "chunks": len(chunks),
        "pages": len(pages),
        "words": len(words),
        "keywords": keywords.most_common(10),
        "document_list": document_list
    }


def get_analytics_report():

    data = get_document_analytics()

    if data["documents"] == 0:
        return "No documents have been processed yet."

    report = f"""
## Document Analytics

- **Total Documents:** {data['documents']}
- **Indexed Pages:** {data['pages']}
- **Total Chunks:** {data['chunks']}
- **Extracted Words:** {data['words']}

### Top Keywords

"""

    for word, count in data["keywords"]:
        report += f"- **{word}** — {count} occurrences\n"

    report += "\n### Document Details\n"

    for doc in data["document_list"]:
        report += (
            f"- **{doc['filename']}** | "
            f"{doc['indexed_pages']} indexed pages | "
            f"{doc['chunks']} chunks\n"
        )

    return report


print("Document Analytics Engine ready!")

print(get_analytics_report())


Document Analytics Engine ready!

## Document Analytics

- **Total Documents:** 1
- **Indexed Pages:** 14
- **Total Chunks:** 37
- **Extracted Words:** 3665

### Top Keywords

- **page** — 54 occurrences
- **text** — 52 occurrences
- **answer** — 48 occurrences
- **chunks** — 39 occurrences
- **question** — 36 occurrences
- **model** — 33 occurrences
- **research** — 32 occurrences
- **flan** — 28 occurrences
- **not** — 24 occurrences
- **paper** — 21 occurrences

### Document Details
- **DocuMind AI — Research Paper Intelligence Studio (1).pdf** | 14 indexed pages | 37 chunks



In [ ]:

import re

def conversational_qa(message, history=None):

    if not message or not message.strip():
        return "Please enter a question."

    history = history or []

    # Find the most recent user question
    previous_question = None

    for item in reversed(history):

        if isinstance(item, dict):

            if item.get("role") == "user":
                previous_question = item.get("content")
                break

        elif isinstance(item, (list, tuple)) and len(item) >= 1:
            # Compatibility with older Gradio history format
            previous_question = item[0]
            if previous_question:
                break

    current_question = message.strip()

    # Detect short or context-dependent follow-up questions
    followup_pattern = (
        r"\b(it|its|they|them|this|that|these|those|"
        r"their|above|same|also|more|explain further|"
        r"what about|how about|why|elaborate)\b"
    )

    is_followup = (
        len(current_question.split()) <= 8
        or re.search(followup_pattern, current_question.lower())
    )

    if previous_question and is_followup:

        resolved_question = (
            f"Previous user question: {previous_question}\n"
            f"Current follow-up question: {current_question}\n"
            "Answer the current follow-up question using the "
            "previous question only as context."
        )

    else:
        resolved_question = current_question

    # Use the existing document Q&A engine
    return universal_qa(resolved_question)


print("Conversational AI Engine ready!")
print("Follow-up question support enabled!")


Conversational AI Engine ready!
Follow-up question support enabled!


In [ ]:

# STEP 4: RESEARCH INSIGHTS ENGINE

INSIGHT_QUESTIONS = {
    "Research Summary":
        "Provide a structured summary of this research paper, "
        "including its topic, problem, approach and conclusion.",

    "Research Objectives":
        "What are the main objectives and research questions "
        "of this paper?",

    "Research Methodology":
        "Explain the research methodology, methods, data "
        "collection and techniques used in this paper.",

    "Key Findings":
        "What are the major findings and results reported "
        "in this research paper?",

    "Research Limitations":
        "What limitations, challenges or weaknesses are "
        "mentioned in this research paper?",

    "Conclusion":
        "Explain the conclusion and key takeaways of this paper.",

    "Future Scope":
        "What future research directions or improvements "
        "are suggested in this paper?"
}


def generate_research_insight(insight_type):

    if not doc_store["chunks"] or doc_store["index"] is None:
        return "Please upload and process a research paper first."

    question = INSIGHT_QUESTIONS.get(insight_type)

    if not question:
        return "Please select a valid research insight."

    answer = universal_qa(question)

    return (
        f"## {insight_type}\n\n"
        f"{answer}"
    )


print("Research Insights Engine ready!")
print("Available insights:")

for item in INSIGHT_QUESTIONS:
    print("-", item)


Research Insights Engine ready!
Available insights:
- Research Summary
- Research Objectives
- Research Methodology
- Key Findings
- Research Limitations
- Conclusion
- Future Scope


In [ ]:
import gradio as gr

print("Gradio version:", gr.__version__)
print("PDF chunks:", len(doc_store["chunks"]))
print("FAISS index:", doc_store["index"] is not None)
print("Analytics:", callable(get_document_analytics))
print("Research Insights:", callable(generate_research_insight))
print("Conversational AI:", callable(conversational_qa))

Gradio version: 6.31.0
PDF chunks: 37
FAISS index: True
Analytics: True
Research Insights: True
Conversational AI: True


In [ ]:

# ============================================================
# DOCUMIND AI - FINAL RESEARCH INTELLIGENCE STUDIO
# Compatible with Gradio 6.31.0
# ============================================================

import os
import re
import sqlite3
import datetime
from collections import Counter

import gradio as gr
import numpy as np
import fitz
import faiss


# ------------------------------------------------------------
# 1. BACKEND VALIDATION
# ------------------------------------------------------------

required_variables = [
    "doc_store",
    "embedding_model",
    "tokenizer",
    "model"
]

missing = [v for v in required_variables if v not in globals()]

if missing:
    raise RuntimeError(
        "Backend variables missing: " + ", ".join(missing)
        + ". Please run the previous backend cells first."
    )

if not isinstance(doc_store, dict):
    raise RuntimeError("doc_store is not initialized correctly.")

if "chunks" not in doc_store:
    doc_store["chunks"] = []


# ------------------------------------------------------------
# 2. DATABASE - SAVED CHAT HISTORY
# ------------------------------------------------------------

DB_PATH = "/content/documind_history.db"

def init_database():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            updated_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id INTEGER,
            role TEXT,
            content TEXT
        )
    """)

    conn.commit()
    conn.close()

init_database()


def get_saved_chats():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        SELECT id, title
        FROM conversations
        ORDER BY updated_at DESC
    """)

    rows = cur.fetchall()
    conn.close()

    return [(title, str(cid)) for cid, title in rows]


def create_chat():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    now = datetime.datetime.now().isoformat()

    cur.execute("""
        INSERT INTO conversations (title, updated_at)
        VALUES (?, ?)
    """, ("New Research Chat", now))

    cid = cur.lastrowid

    conn.commit()
    conn.close()

    return cid


def save_chat_message(cid, role, content):
    if cid is None:
        return

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO messages (conversation_id, role, content)
        VALUES (?, ?, ?)
    """, (int(cid), role, content))

    now = datetime.datetime.now().isoformat()

    cur.execute("""
        UPDATE conversations
        SET updated_at = ?
        WHERE id = ?
    """, (now, int(cid)))

    cur.execute("""
        SELECT content
        FROM messages
        WHERE conversation_id = ? AND role = 'user'
        ORDER BY id ASC
        LIMIT 1
    """, (int(cid),))

    first_question = cur.fetchone()

    if first_question:
        title = first_question[0][:45]
        cur.execute("""
            UPDATE conversations
            SET title = ?
            WHERE id = ?
        """, (title, int(cid)))

    conn.commit()
    conn.close()


def load_chat_messages(cid):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id ASC
    """, (int(cid),))

    rows = cur.fetchall()
    conn.close()

    return [
        {"role": role, "content": content}
        for role, content in rows
    ]


# ------------------------------------------------------------
# 3. DOCUMENT HELPERS
# ------------------------------------------------------------

def chunk_value(chunk, keys, default=""):
    for key in keys:
        if isinstance(chunk, dict) and key in chunk:
            return chunk[key]
    return default


def get_documents():
    names = set()

    for chunk in doc_store["chunks"]:
        name = chunk_value(
            chunk,
            ["source", "filename", "file_name", "document"],
            "Unknown PDF"
        )
        names.add(str(name))

    return sorted(names)


def rebuild_search_index(chunks):
    if not chunks:
        doc_store["index"] = None
        return

    texts = [
        str(chunk_value(c, ["text", "content"], ""))
        for c in chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    embeddings = np.asarray(embeddings, dtype="float32")
    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    doc_store["index"] = index


def refresh_document_choices():
    return gr.update(choices=get_documents(), value=None)


# ------------------------------------------------------------
# 4. MULTIPLE PDF UPLOAD
# ------------------------------------------------------------

def upload_documents(files):
    if not files:
        return "Please select at least one PDF.", gr.update(
            choices=get_documents()
        )

    existing = set(get_documents())
    added = []
    skipped = []
    new_chunks = []

    for file_path in files:
        if not file_path or not os.path.exists(file_path):
            continue

        filename = os.path.basename(file_path)

        if filename in existing:
            skipped.append(filename)
            continue

        try:
            pdf = fitz.open(file_path)

            for page_number, page in enumerate(pdf, start=1):
                page_text = page.get_text("text").strip()

                if not page_text:
                    continue

                # Split page text into overlapping chunks
                chunk_size = 1200
                overlap = 200
                start = 0

                while start < len(page_text):
                    end = start + chunk_size
                    text_part = page_text[start:end].strip()

                    if text_part:
                        new_chunks.append({
                            "text": text_part,
                            "source": filename,
                            "page": page_number
                        })

                    start += chunk_size - overlap

            pdf.close()
            added.append(filename)
            existing.add(filename)

        except Exception as e:
            skipped.append(filename + " (Error: " + str(e) + ")")

    if new_chunks:
        doc_store["chunks"].extend(new_chunks)
        rebuild_search_index(doc_store["chunks"])

    status = "## PDF Processing Report\n\n"

    status += "**Successfully added:** " + str(len(added)) + "\n\n"

    if added:
        status += "\n".join("- " + x for x in added) + "\n\n"

    status += "**Skipped / already uploaded / errors:** " + str(len(skipped))

    if skipped:
        status += "\n\n" + "\n".join("- " + x for x in skipped)

    status += "\n\n**Total indexed chunks:** " + str(
        len(doc_store["chunks"])
    )

    return status, gr.update(choices=get_documents())


# ------------------------------------------------------------
# 5. DELETE DOCUMENT
# ------------------------------------------------------------

def delete_document(name):
    if not name:
        return (
            "Please select a PDF to delete.",
            gr.update(choices=get_documents(), value=None)
        )

    before = len(doc_store["chunks"])

    doc_store["chunks"] = [
        c for c in doc_store["chunks"]
        if str(chunk_value(
            c,
            ["source", "filename", "file_name", "document"],
            "Unknown PDF"
        )) != str(name)
    ]

    removed = before - len(doc_store["chunks"])

    rebuild_search_index(doc_store["chunks"])

    status = (
        f"Deleted **{name}** successfully.\n\n"
        f"Removed chunks: {removed}"
    )

    return status, gr.update(choices=get_documents(), value=None)


# ------------------------------------------------------------
# 6. DOCUMENT QUESTION ANSWERING ENGINE
# ------------------------------------------------------------

def answer_from_documents(question, selected_docs=None, history=None):

    question = str(question).strip()

    if not question:
        return "Please enter a question."

    chunks = doc_store["chunks"]

    if selected_docs:
        selected_docs = [str(x) for x in selected_docs]

        chunks = [
            c for c in chunks
            if str(chunk_value(
                c,
                ["source", "filename", "file_name", "document"],
                "Unknown PDF"
            )) in selected_docs
        ]

    if not chunks:
        return "No indexed PDF content is available for the selected documents."

    # Retrieve relevant chunks
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    faiss.normalize_L2(query_embedding)

    chunk_texts = [
        str(chunk_value(c, ["text", "content"], ""))
        for c in chunks
    ]

    document_embeddings = embedding_model.encode(
        chunk_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    document_embeddings = np.asarray(
        document_embeddings,
        dtype="float32"
    )

    faiss.normalize_L2(document_embeddings)

    similarities = np.dot(
        document_embeddings,
        query_embedding[0]
    )

    top_indices = np.argsort(similarities)[::-1][:6]

    relevant_chunks = [
        chunks[i]
        for i in top_indices
        if similarities[i] > 0.05
    ]

    if not relevant_chunks:
        return (
            "I could not find sufficiently relevant information "
            "in the uploaded research papers."
        )

    context_parts = []
    sources = []

    for c in relevant_chunks:
        source = str(chunk_value(
            c,
            ["source", "filename", "file_name", "document"],
            "Unknown PDF"
        ))

        page = chunk_value(c, ["page", "page_number"], "?")
        text_part = str(chunk_value(c, ["text", "content"], ""))

        context_parts.append(
            f"Source: {source}, Page: {page}\n{text_part}"
        )

        sources.append(f"{source} (Page {page})")

    context = "\n\n".join(context_parts)
    context = context[:3500]

    prompt = f"""
You are DocuMind AI, an academic research paper assistant.

Answer the user's question using the research paper content below.

Instructions:
1. Give a meaningful and explanatory answer.
2. Use simple, clear academic English.
3. Explain the important points in a connected paragraph.
4. Avoid one-word or extremely short answers.
5. Use bullet points only when they improve clarity.
6. Do not invent facts that are not present in the papers.
7. If the information is not available, clearly say so.
8. Focus on the exact question asked.

Research paper content:
{context}

Question:
{question}

Detailed answer:
"""

    try:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        device = next(model.parameters()).device

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            num_beams=4,
            early_stopping=True
        )

        answer = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()

    except Exception as e:
        return "AI generation error: " + str(e)

    if len(answer.split()) < 12:
        answer += (
            "\n\nThe retrieved paper content contains limited "
            "information about this question."
        )

    unique_sources = list(dict.fromkeys(sources))

    answer += "\n\n---\n**Sources consulted:**\n"
    answer += "\n".join("- " + s for s in unique_sources)

    return answer


# ------------------------------------------------------------
# 7. CHAT OPERATIONS
# ------------------------------------------------------------

def send_chat(message, history, selected_docs, cid):

    if not message or not str(message).strip():
        return "", history or [], cid, gr.update(
            choices=get_saved_chats()
        )

    if cid is None:
        cid = create_chat()

    history = history or []

    save_chat_message(cid, "user", message)

    answer = answer_from_documents(
        message,
        selected_docs,
        history
    )

    save_chat_message(cid, "assistant", answer)

    updated_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]

    return (
        "",
        updated_history,
        cid,
        gr.update(choices=get_saved_chats(), value=str(cid))
    )


def new_chat():
    cid = create_chat()

    return (
        cid,
        [],
        gr.update(choices=get_saved_chats(), value=str(cid))
    )


def open_chat(cid):
    if not cid:
        return [], None

    messages = load_chat_messages(cid)

    return messages, int(cid)


def remove_chat(cid):
    if not cid:
        return [], None, gr.update(choices=get_saved_chats())

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute(
        "DELETE FROM messages WHERE conversation_id = ?",
        (int(cid),)
    )

    cur.execute(
        "DELETE FROM conversations WHERE id = ?",
        (int(cid),)
    )

    conn.commit()
    conn.close()

    return (
        [],
        None,
        gr.update(choices=get_saved_chats(), value=None)
    )


# ------------------------------------------------------------
# 8. RESEARCH INSIGHTS ENGINE
# ------------------------------------------------------------

INSIGHTS = [
    "Research Summary",
    "Research Objectives",
    "Research Methodology",
    "Key Findings",
    "Research Limitations",
    "Conclusion",
    "Future Scope",
    "Research Problem",
    "Research Contribution"
]


def make_insight(choice, selected_docs):

    prompts = {
        "Research Summary":
            "Provide a detailed summary of this research paper.",

        "Research Objectives":
            "Explain the main objectives of this research.",

        "Research Methodology":
            "Explain the research methodology, methods and techniques used.",

        "Key Findings":
            "Explain the important findings and results of this research.",

        "Research Limitations":
            "Identify and explain the limitations of this research.",

        "Conclusion":
            "Explain the conclusion of this research paper.",

        "Future Scope":
            "Explain the possible future research directions.",

        "Research Problem":
            "What research problem is addressed in this paper? Explain it.",

        "Research Contribution":
            "Explain the major contributions and novelty of this research."
    }

    prompt = prompts.get(choice)

    if not prompt:
        return "Please select a valid research insight."

    return answer_from_documents(prompt, selected_docs)


# ------------------------------------------------------------
# 9. DOCUMENT ANALYTICS
# ------------------------------------------------------------

def analytics_report():

    chunks = doc_store["chunks"]

    if not chunks:
        return "No documents indexed yet."

    documents = get_documents()

    all_text = " ".join([
        str(chunk_value(c, ["text", "content"], ""))
        for c in chunks
    ])

    words = re.findall(r"\b[a-zA-Z]{3,}\b", all_text.lower())

    stop_words = {
        "the", "and", "for", "with", "this", "that",
        "from", "are", "was", "were", "have", "has",
        "into", "their", "they", "which", "using",
        "can", "will", "not", "also", "been", "its",
        "our", "but", "such", "than", "then"
    }

    filtered_words = [
        w for w in words if w not in stop_words
    ]

    top_words = Counter(filtered_words).most_common(15)

    page_count = len(set(
        (
            str(chunk_value(c, ["source", "filename"], "Unknown")),
            str(chunk_value(c, ["page", "page_number"], "?"))
        )
        for c in chunks
    ))

    report = "# Document Analytics Report\n\n"

    report += f"**Total Documents:** {len(documents)}\n\n"
    report += f"**Indexed Pages:** {page_count}\n\n"
    report += f"**Total Chunks:** {len(chunks)}\n\n"
    report += f"**Words in Indexed Chunks:** {len(words)}\n\n"

    report += "## Top Research Keywords\n\n"

    for word, count in top_words:
        report += f"- **{word}** — {count} occurrences\n"

    report += "\n## Uploaded Documents\n\n"

    for name in documents:
        report += f"- {name}\n"

    return report


# ------------------------------------------------------------
# 10. DASHBOARD METRICS
# ------------------------------------------------------------

def get_metrics():

    chunks = doc_store["chunks"]
    documents = get_documents()

    pages = len(set(
        (
            str(chunk_value(c, ["source", "filename"], "Unknown")),
            str(chunk_value(c, ["page", "page_number"], "?"))
        )
        for c in chunks
    ))

    words = sum(
        len(str(chunk_value(c, ["text", "content"], "")).split())
        for c in chunks
    )

    return (
        str(len(documents)),
        str(pages),
        str(len(chunks)),
        str(words)
    )


# ------------------------------------------------------------
# 11. PRESENTATION CONTENT
# ------------------------------------------------------------

presentation = """
# DOCUMIND AI
## Research Paper Intelligence Platform

### 1. Project Objective
To develop an AI-powered platform that helps users understand,
analyze and retrieve information from research papers efficiently.

### 2. Problem Statement
Research papers are often lengthy and technically complex.
Manually finding specific information takes time and effort.

### 3. Proposed Solution
DocuMind AI allows users to upload research PDFs, ask natural
language questions, explore research insights and view document analytics.

### 4. System Workflow
1. Upload research PDF files.
2. Extract text from PDF pages.
3. Split extracted text into smaller chunks.
4. Generate semantic embeddings.
5. Store and retrieve relevant chunks.
6. Generate answers using an AI language model.
7. Display answers with source and page references.

### 5. Technology Stack
- Python
- Google Colab
- Gradio
- PyMuPDF
- Sentence Transformers
- FAISS
- FLAN-T5
- NumPy
- SQLite

### 6. Main Modules
- PDF Processing and Document Management
- Semantic Search and Retrieval
- AI Research Assistant
- Research Insights Generator
- Document Analytics
- Chat History Management

### 7. Demonstration Sequence
1. Upload one or more research papers.
2. Show document indexing.
3. Ask questions about the uploaded papers.
4. Generate research insights.
5. Explore document analytics.
6. Reopen a saved conversation.

### 8. Current Limitations
- Answers depend on the text successfully extracted from PDFs.
- Scanned PDFs may require OCR.
- Generated answers should be verified against original papers.
- Chat database is stored in the current Colab runtime.

### 9. Future Enhancements
- OCR support for scanned documents.
- Advanced multi-paper comparison.
- Export research summaries.
- Persistent cloud-based document storage.
- More advanced language models.
"""


# ------------------------------------------------------------
# 12. PROFESSIONAL UI DESIGN
# ------------------------------------------------------------

css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

.gradio-container {
    background: #f5f8f7 !important;
    font-family: 'Inter', sans-serif !important;
}

#main-header {
    background: linear-gradient(120deg, #102a35, #14545a);
    color: white;
    padding: 30px;
    border-radius: 18px;
    margin-bottom: 18px;
}

#main-header h1 {
    font-size: 34px;
    font-weight: 800;
    margin-bottom: 6px;
}

#main-header p {
    color: #d2e8e5;
    font-size: 14px;
}

.metric-card {
    background: white;
    border: 1px solid #dce8e5;
    border-radius: 14px;
    padding: 18px;
    text-align: center;
}

.metric-number {
    color: #087f78;
    font-size: 27px;
    font-weight: 800;
}

.metric-label {
    color: #53666a;
    font-size: 12px;
    font-weight: 600;
    text-transform: uppercase;
}

.section-title {
    color: #173b43;
    font-weight: 700;
}

button.primary {
    background: #087f78 !important;
    border-color: #087f78 !important;
}

footer {
    display: none !important;
}
"""


# ------------------------------------------------------------
# 13. BUILD GRADIO APPLICATION
# ------------------------------------------------------------

with gr.Blocks(title="DocuMind AI - Research Studio") as demo:

    chat_id_state = gr.State(value=None)

    gr.HTML("""
    <div id="main-header">
        <h1>DOCUMIND AI</h1>
        <p>RESEARCH PAPER INTELLIGENCE STUDIO</p>
        <p>Upload • Understand • Analyze • Discover</p>
    </div>
    """)

    # Dashboard metrics

    with gr.Row():
        with gr.Column():
            gr.HTML("""
            <div class="metric-card">
                <div class="metric-label">Documents</div>
                <div class="metric-number" id="metric-docs">PDF</div>
            </div>
            """)
            m_docs = gr.Textbox(label="Total Documents", interactive=False)

        with gr.Column():
            gr.HTML("""
            <div class="metric-card">
                <div class="metric-label">Pages</div>
                <div class="metric-number">PDF</div>
            </div>
            """)
            m_pages = gr.Textbox(label="Indexed Pages", interactive=False)

        with gr.Column():
            gr.HTML("""
            <div class="metric-card">
                <div class="metric-label">Chunks</div>
                <div class="metric-number">AI</div>
            </div>
            """)
            m_chunks = gr.Textbox(label="Total Chunks", interactive=False)

        with gr.Column():
            gr.HTML("""
            <div class="metric-card">
                <div class="metric-label">Words</div>
                <div class="metric-number">NLP</div>
            </div>
            """)
            m_words = gr.Textbox(label="Indexed Words", interactive=False)


    with gr.Tabs():

        # ----------------------------------------------------
        # TAB 1 - RESEARCH LIBRARY
        # ----------------------------------------------------

        with gr.Tab("01 | Research Library"):

            gr.Markdown("## Manage Research Documents")

            gr.Markdown(
                "Upload multiple research papers in PDF format. "
                "The system extracts text and indexes the content."
            )

            pdf_upload = gr.File(
                label="Upload Research PDFs",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            upload_button = gr.Button(
                "Upload and Index Documents",
                variant="primary"
            )

            upload_status = gr.Markdown()

            gr.Markdown("### Uploaded Document Collection")

            document_list = gr.Dropdown(
                label="Select PDF to Delete",
                choices=get_documents()
            )

            delete_button = gr.Button(
                "Delete Selected Document",
                variant="stop"
            )

            delete_status = gr.Markdown()

            upload_button.click(
                fn=upload_documents,
                inputs=[pdf_upload],
                outputs=[upload_status, document_list]
            ).then(
                fn=get_metrics,
                outputs=[m_docs, m_pages, m_chunks, m_words]
            )

            delete_button.click(
                fn=delete_document,
                inputs=[document_list],
                outputs=[delete_status, document_list]
            ).then(
                fn=get_metrics,
                outputs=[m_docs, m_pages, m_chunks, m_words]
            )


        # ----------------------------------------------------
        # TAB 2 - AI RESEARCH ASSISTANT
        # ----------------------------------------------------

        with gr.Tab("02 | AI Research Assistant"):

            gr.Markdown("## Ask Questions About Your Research Papers")

            document_scope = gr.Dropdown(
                label="Choose Documents (Empty = All Documents)",
                choices=get_documents(),
                multiselect=True
            )

            chatbot = gr.Chatbot(
                label="Research Conversation",
                height=480
            )

            with gr.Row():
                message_box = gr.Textbox(
                    label="Ask your research question",
                    placeholder="Example: Explain the methodology used in this paper...",
                    scale=5
                )

                send_button = gr.Button(
                    "Send",
                    variant="primary",
                    scale=1
                )

            with gr.Row():
                new_chat_button = gr.Button("New Chat")
                saved_chat_dropdown = gr.Dropdown(
                    label="Saved Conversations",
                    choices=get_saved_chats()
                )
                open_chat_button = gr.Button("Open Chat")
                delete_chat_button = gr.Button(
                    "Delete Chat",
                    variant="stop"
                )

            send_button.click(
                fn=send_chat,
                inputs=[
                    message_box,
                    chatbot,
                    document_scope,
                    chat_id_state
                ],
                outputs=[
                    message_box,
                    chatbot,
                    chat_id_state,
                    saved_chat_dropdown
                ]
            )

            message_box.submit(
                fn=send_chat,
                inputs=[
                    message_box,
                    chatbot,
                    document_scope,
                    chat_id_state
                ],
                outputs=[
                    message_box,
                    chatbot,
                    chat_id_state,
                    saved_chat_dropdown
                ]
            )

            new_chat_button.click(
                fn=new_chat,
                outputs=[
                    chat_id_state,
                    chatbot,
                    saved_chat_dropdown
                ]
            )

            open_chat_button.click(
                fn=open_chat,
                inputs=[saved_chat_dropdown],
                outputs=[chatbot, chat_id_state]
            )

            delete_chat_button.click(
                fn=remove_chat,
                inputs=[chat_id_state],
                outputs=[
                    chatbot,
                    chat_id_state,
                    saved_chat_dropdown
                ]
            )


        # ----------------------------------------------------
        # TAB 3 - RESEARCH INSIGHTS
        # ----------------------------------------------------

        with gr.Tab("03 | Research Insights"):

            gr.Markdown("## AI-Powered Research Insights")

            insight_scope = gr.Dropdown(
                label="Select Research Documents (Empty = All)",
                choices=get_documents(),
                multiselect=True
            )

            insight_type = gr.Dropdown(
                label="Select Insight Type",
                choices=INSIGHTS,
                value="Research Summary"
            )

            insight_button = gr.Button(
                "Generate Research Insight",
                variant="primary"
            )

            insight_output = gr.Markdown()

            insight_button.click(
                fn=make_insight,
                inputs=[insight_type, insight_scope],
                outputs=[insight_output]
            )


        # ----------------------------------------------------
        # TAB 4 - ANALYTICS
        # ----------------------------------------------------

        with gr.Tab("04 | Document Analytics"):

            gr.Markdown("## Document Analytics Dashboard")

            analytics_button = gr.Button(
                "Generate Analytics Report",
                variant="primary"
            )

            analytics_output = gr.Markdown()

            analytics_button.click(
                fn=analytics_report,
                outputs=[analytics_output]
            )


        # ----------------------------------------------------
        # TAB 5 - PROJECT PRESENTATION
        # ----------------------------------------------------

        with gr.Tab("05 | Project Presentation"):

            gr.Markdown(presentation)


    gr.Markdown(
        "---\n"
        "<center>DOCUMIND AI | Research Intelligence Platform</center>"
    )

    # IMPORTANT: load must stay INSIDE the Blocks context
    demo.load(
        fn=get_metrics,
        inputs=None,
        outputs=[
            m_docs,
            m_pages,
            m_chunks,
            m_words
        ]
    )


# ------------------------------------------------------------
# 14. LAUNCH APPLICATION
# ------------------------------------------------------------

print("DocuMind AI Research Studio is starting...")

demo.launch(
    share=False,
    debug=False,
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="cyan",
        neutral_hue="slate"
    ),
    css=css
)


DocuMind AI Research Studio is starting...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
for name in ["doc_store", "embedding_model", "tokenizer", "model"]:
    print(name, ":", name in globals())

print("Chunks:", len(doc_store["chunks"]))
print("FAISS vectors:", doc_store["index"].ntotal)

doc_store : True
embedding_model : True
tokenizer : True
model : True
Chunks: 37
FAISS vectors: 37


In [ ]:
import builtins

for name in ["doc_store", "embedding_model", "tokenizer", "model"]:
    print(f"{name}: ", name in globals())

doc_store:  True
embedding_model:  True
tokenizer:  True
model:  True


In [ ]:
# ============================================================
# DOCUMIND AI - COMPLETE COMPATIBLE GUI
# ============================================================

import os
import re
import sqlite3
import inspect
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
import torch

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf

from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ============================================================
# 1. LOAD AI MODELS
# ============================================================

print("Loading AI models... Please wait.")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

model.eval()

print("AI models loaded successfully!")


# ============================================================
# 2. DATABASE
# ============================================================

conn = sqlite3.connect(
    "documind_history.db",
    check_same_thread=False
)

conn.execute("""
CREATE TABLE IF NOT EXISTS chat_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT,
    answer TEXT
)
""")

conn.commit()


# ============================================================
# 3. DOCUMENT STORAGE
# ============================================================

documents = {}
chunks = []
chunk_metadata = []
index = None


# ============================================================
# 4. TEXT CHUNKING
# ============================================================

def split_text(text, chunk_size=700, overlap=120):

    words = text.split()

    result = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        piece = " ".join(words[start:end])

        if piece.strip():
            result.append(piece)

        start += chunk_size - overlap

    return result


# ============================================================
# 5. PDF PROCESSING
# ============================================================

def process_pdf(file_path):

    filename = os.path.basename(file_path)

    pdf = pymupdf.open(file_path)

    extracted_pages = []

    for page_number, page in enumerate(pdf):

        page_text = page.get_text("text")

        if page_text.strip():

            extracted_pages.append({
                "page": page_number + 1,
                "text": page_text
            })

    pdf.close()

    documents[filename] = extracted_pages

    return filename, len(extracted_pages)


# ============================================================
# 6. VECTOR DATABASE
# ============================================================

def rebuild_vector_database():

    global chunks, chunk_metadata, index

    chunks = []
    chunk_metadata = []

    for filename, pages in documents.items():

        for page_data in pages:

            page_chunks = split_text(
                page_data["text"]
            )

            for chunk in page_chunks:

                chunks.append(chunk)

                chunk_metadata.append({
                    "document": filename,
                    "page": page_data["page"]
                })

    if not chunks:

        index = None

        return

    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32
    )

    faiss_index = __import__("faiss")

    faiss_index.normalize_L2(embeddings)

    index = faiss_index.IndexFlatIP(
        embeddings.shape[1]
    )

    index.add(embeddings)

    print(
        "Vector database ready:",
        index.ntotal,
        "vectors"
    )


# ============================================================
# 7. DOCUMENT UPLOAD
# ============================================================

def get_library():

    if not documents:
        return "No documents uploaded yet."

    output = "## Uploaded Research Papers\n\n"

    for filename, pages in documents.items():

        output += (
            f"- **{filename}** — "
            f"{len(pages)} pages\n"
        )

    return output


def upload_documents(files):

    if not files:

        return (
            "Please upload PDF files first.",
            get_library()
        )

    uploaded = []

    for file_path in files:

        try:

            filename, page_count = process_pdf(
                file_path
            )

            uploaded.append(
                f"- **{filename}** — {page_count} pages"
            )

        except Exception as error:

            uploaded.append(
                f"- Error: {str(error)}"
            )

    rebuild_vector_database()

    status = (
        "## PDF Processing Completed\n\n"
        + "\n".join(uploaded)
        + f"\n\n**Total Documents:** {len(documents)}"
        + f"\n\n**Total Chunks:** {len(chunks)}"
    )

    return status, get_library()


# ============================================================
# 8. DELETE DOCUMENT
# ============================================================

def delete_document(filename):

    if not filename:

        return (
            "Please select a document.",
            get_library()
        )

    if filename in documents:

        del documents[filename]

        rebuild_vector_database()

        return (
            f"Successfully deleted: {filename}",
            get_library()
        )

    return (
        "Document not found.",
        get_library()
    )


# ============================================================
# 9. SEMANTIC SEARCH
# ============================================================

def retrieve_context(
    question,
    scope="All Documents",
    top_k=5
):

    if index is None or not chunks:
        return []

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    faiss_index = __import__("faiss")

    faiss_index.normalize_L2(
        query_embedding
    )

    scores, indices = index.search(
        query_embedding,
        min(top_k, len(chunks))
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < 0:
            continue

        metadata = chunk_metadata[idx]

        if (
            scope != "All Documents"
            and metadata["document"] != scope
        ):
            continue

        results.append({
            "text": chunks[idx],
            "document": metadata["document"],
            "page": metadata["page"],
            "score": float(score)
        })

    return results


# ============================================================
# 10. AI ANSWER GENERATION
# ============================================================

def generate_answer(
    question,
    scope="All Documents"
):

    if not documents:

        return (
            "Please upload a research paper first.",
            ""
        )

    retrieved = retrieve_context(
        question,
        scope,
        top_k=5
    )

    if not retrieved:

        return (
            "No relevant information was found "
            "in the selected research papers.",
            ""
        )

    context = "\n\n".join(
        item["text"]
        for item in retrieved
    )

    prompt = f"""
You are DocuMind AI, a research paper assistant.

Answer the question using only the context below.

Instructions:
1. Give a clear and detailed answer.
2. Use simple academic English.
3. Explain important concepts properly.
4. Do not invent information.
5. If the answer is unavailable, clearly mention it.

Research Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    sources = "\n".join(
        [
            f"- {item['document']} — Page {item['page']}"
            for item in retrieved
        ]
    )

    return answer, sources


# ============================================================
# 11. CHAT FUNCTION
# ============================================================

def chat_with_ai(
    message,
    history,
    scope
):

    if not message or not message.strip():

        return history or [], ""

    answer, sources = generate_answer(
        message,
        scope
    )

    final_answer = (
        answer
        + "\n\n**Sources:**\n"
        + sources
    )

    history = history or []

    history.append(
        (message, final_answer)
    )

    conn.execute(
        """
        INSERT INTO chat_history (question, answer)
        VALUES (?, ?)
        """,
        (message, final_answer)
    )

    conn.commit()

    return history, ""


def clear_chat():

    return []


# ============================================================
# 12. SAVED CONVERSATIONS
# ============================================================

def get_saved_conversations():

    rows = conn.execute("""
        SELECT id, question, answer
        FROM chat_history
        ORDER BY id DESC
        LIMIT 30
    """).fetchall()

    if not rows:

        return "No saved conversations yet."

    output = "## Saved Conversations\n\n"

    for row in rows:

        output += (
            f"### Question: {row[1]}\n\n"
            f"{row[2]}\n\n"
            "---\n\n"
        )

    return output


def clear_saved_conversations():

    conn.execute(
        "DELETE FROM chat_history"
    )

    conn.commit()

    return "Conversation history cleared successfully."


# ============================================================
# 13. RESEARCH INSIGHTS
# ============================================================

def generate_insights(scope):

    selected_docs = documents

    if scope != "All Documents":

        selected_docs = {
            scope: documents.get(scope, [])
        }

    if not selected_docs:

        return "Please upload a research paper first."

    all_text = []

    total_pages = 0

    for filename, pages in selected_docs.items():

        total_pages += len(pages)

        for page in pages:

            all_text.append(
                page["text"]
            )

    combined_text = " ".join(all_text)

    words = re.findall(
        r"\b[a-zA-Z]{3,}\b",
        combined_text.lower()
    )

    stopwords = {
        "the", "and", "for", "are", "was",
        "with", "that", "this", "from", "have",
        "has", "were", "been", "their", "they",
        "which", "will", "would", "could", "should",
        "into", "also", "than", "then", "when",
        "where", "what", "how", "can", "may",
        "not", "but", "all", "any", "its",
        "our", "out", "use", "used", "using"
    }

    filtered_words = [
        word for word in words
        if word not in stopwords
    ]

    counts = Counter(filtered_words)

    top_keywords = counts.most_common(15)

    output = "## Research Insights\n\n"

    output += (
        f"**Documents Analysed:** "
        f"{len(selected_docs)}\n\n"
    )

    output += (
        f"**Total Pages:** {total_pages}\n\n"
    )

    output += (
        f"**Total Words:** {len(words)}\n\n"
    )

    output += (
        f"**Unique Keywords:** {len(counts)}\n\n"
    )

    output += "### Top Research Keywords\n\n"

    for word, count in top_keywords:

        output += (
            f"- **{word}** — "
            f"{count} occurrences\n"
        )

    return output


# ============================================================
# 14. ANALYTICS GRAPHS
# ============================================================

def document_distribution():

    if not documents:
        return None

    names = list(documents.keys())

    page_counts = [
        len(documents[name])
        for name in names
    ]

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.bar(
        names,
        page_counts
    )

    ax.set_title(
        "Document Page Distribution"
    )

    ax.set_xlabel("Documents")

    ax.set_ylabel("Number of Pages")

    plt.xticks(
        rotation=30,
        ha="right"
    )

    plt.tight_layout()

    return fig


def keyword_analysis():

    if not documents:
        return None

    all_text = " ".join(
        page["text"]
        for pages in documents.values()
        for page in pages
    )

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        all_text.lower()
    )

    stopwords = {
        "this", "that", "with", "from",
        "have", "were", "which", "their",
        "there", "these", "those", "been",
        "also", "into", "using", "used"
    }

    counts = Counter(
        word
        for word in words
        if word not in stopwords
    )

    top = counts.most_common(10)

    if not top:
        return None

    labels = [
        item[0] for item in top
    ]

    values = [
        item[1] for item in top
    ]

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.barh(
        labels[::-1],
        values[::-1]
    )

    ax.set_title(
        "Top Research Keywords"
    )

    ax.set_xlabel("Frequency")

    plt.tight_layout()

    return fig


def page_distribution():

    if not documents:
        return None

    names = list(documents.keys())

    sizes = [
        len(documents[name])
        for name in names
    ]

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.pie(
        sizes,
        labels=names,
        autopct="%1.1f%%",
        startangle=90
    )

    ax.set_title(
        "Document Collection Distribution"
    )

    plt.tight_layout()

    return fig


def refresh_analytics():

    return (
        document_distribution(),
        keyword_analysis(),
        page_distribution()
    )


# ============================================================
# 15. PROJECT OVERVIEW
# ============================================================

def project_overview():

    total_pages = sum(
        len(pages)
        for pages in documents.values()
    )

    total_words = sum(
        len(page["text"].split())
        for pages in documents.values()
        for page in pages
    )

    return f"""
# 📚 DocuMind AI

### Research Paper Intelligence Studio

DocuMind AI is an AI-powered research paper
analysis and question-answering application.

## Project Statistics

- **Total Documents:** {len(documents)}
- **Indexed Pages:** {total_pages}
- **Total Chunks:** {len(chunks)}
- **Extracted Words:** {total_words}

## Technologies Used

- Python
- Gradio
- Sentence Transformers
- FAISS
- FLAN-T5
- PyMuPDF
- SQLite
- Matplotlib

## Key Features

1. Multiple PDF Upload
2. Document Management
3. Semantic Search
4. AI Research Assistant
5. Source-based Answers
6. Research Insights
7. Visual Analytics
8. Saved Conversations
"""


# ============================================================
# 16. GRADIO CHATBOT COMPATIBILITY
# ============================================================

# Automatically detect whether this Gradio version
# supports the "type" argument in Chatbot.

chatbot_parameters = inspect.signature(
    gr.Chatbot.__init__
).parameters

chatbot_options = {
    "label": "💬 Research Conversation",
    "height": 460
}

if "type" in chatbot_parameters:

    chatbot_options["type"] = "tuples"


# ============================================================
# 17. THEME AND CSS
# ============================================================

theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="purple",
    neutral_hue="slate"
)

custom_css = """
.gradio-container {
    background: linear-gradient(
        135deg,
        #fff7fb,
        #f4edff
    );
}

h1, h2, h3 {
    color: #9d2878 !important;
}

button.primary {
    background: linear-gradient(
        90deg,
        #d63384,
        #8b5cf6
    ) !important;

    color: white !important;
}
"""


# ============================================================
# 18. COMPLETE GUI
# ============================================================

with gr.Blocks(
    title="DocuMind AI - Research Intelligence Studio",
    theme=theme,
    css=custom_css
) as demo:

    gr.Markdown("""
    # 📚 DocuMind AI
    ### Research Paper Intelligence Studio

    Upload research papers, ask questions and explore
    insights using AI-powered semantic search.
    """)

    with gr.Tabs():

        # ----------------------------------------------------
        # TAB 1 - RESEARCH LIBRARY
        # ----------------------------------------------------

        with gr.Tab("📚 Research Library"):

            gr.Markdown(
                "## Manage Research Documents"
            )

            pdf_upload = gr.File(
                label="Upload Research Paper PDFs",
                file_types=[".pdf"],
                file_count="multiple",
                type="filepath"
            )

            upload_button = gr.Button(
                "📤 Upload and Process",
                variant="primary"
            )

            upload_status = gr.Markdown()

            library_output = gr.Markdown(
                value=get_library()
            )

            delete_dropdown = gr.Dropdown(
                choices=[],
                label="Select Document to Delete"
            )

            delete_button = gr.Button(
                "🗑️ Delete Selected Document"
            )

            delete_status = gr.Markdown()


        # ----------------------------------------------------
        # TAB 2 - AI RESEARCH ASSISTANT
        # ----------------------------------------------------

        with gr.Tab("🤖 AI Research Assistant"):

            gr.Markdown("""
            ## Ask Questions About Your Research Papers

            Ask questions in natural language and receive
            AI-generated answers with document references.
            """)

            document_scope = gr.Dropdown(
                choices=["All Documents"],
                value="All Documents",
                label="Select Research Document"
            )

            chatbot = gr.Chatbot(
                **chatbot_options
            )

            question_input = gr.Textbox(
                label="Enter Your Research Question",
                placeholder=(
                    "Example: What is the main objective "
                    "of this research paper?"
                ),
                lines=2
            )

            with gr.Row():

                send_button = gr.Button(
                    "🚀 Ask AI",
                    variant="primary"
                )

                clear_chat_button = gr.Button(
                    "Clear Chat"
                )


        # ----------------------------------------------------
        # TAB 3 - RESEARCH INSIGHTS
        # ----------------------------------------------------

        with gr.Tab("✨ Research Insights"):

            gr.Markdown(
                "## Research Insights Dashboard"
            )

            insight_scope = gr.Dropdown(
                choices=["All Documents"],
                value="All Documents",
                label="Select Document"
            )

            insight_button = gr.Button(
                "Generate Research Insights",
                variant="primary"
            )

            insight_output = gr.Markdown()


        # ----------------------------------------------------
        # TAB 4 - VISUAL ANALYTICS
        # ----------------------------------------------------

        with gr.Tab("📊 Visual Analytics"):

            gr.Markdown(
                "## Research Analytics Dashboard"
            )

            refresh_btn = gr.Button(
                "Refresh Analytics",
                variant="primary"
            )

            with gr.Row():

                document_plot = gr.Plot(
                    label="Document Distribution"
                )

                keyword_plot = gr.Plot(
                    label="Keyword Analysis"
                )

            page_plot = gr.Plot(
                label="Document Collection Distribution"
            )


        # ----------------------------------------------------
        # TAB 5 - SAVED CONVERSATIONS
        # ----------------------------------------------------

        with gr.Tab("💾 Saved Conversations"):

            gr.Markdown(
                "## Conversation History"
            )

            history_button = gr.Button(
                "Load Saved Conversations"
            )

            history_output = gr.Markdown()

            clear_history_button = gr.Button(
                "Clear Saved History"
            )

            history_status = gr.Markdown()


        # ----------------------------------------------------
        # TAB 6 - PROJECT OVERVIEW
        # ----------------------------------------------------

        with gr.Tab("ℹ️ Project Overview"):

            overview_output = gr.Markdown(
                value=project_overview()
            )


    # ========================================================
    # 19. CALLBACKS
    # ========================================================

    def upload_callback(files):

        status, library = upload_documents(files)

        choices = (
            ["All Documents"]
            + list(documents.keys())
        )

        return (
            status,
            library,
            gr.update(
                choices=choices,
                value="All Documents"
            ),
            gr.update(
                choices=choices,
                value="All Documents"
            ),
            gr.update(
                choices=list(documents.keys()),
                value=None
            ),
            project_overview()
        )


    def delete_callback(filename):

        status, library = delete_document(filename)

        choices = (
            ["All Documents"]
            + list(documents.keys())
        )

        return (
            status,
            library,
            gr.update(
                choices=choices,
                value="All Documents"
            ),
            gr.update(
                choices=choices,
                value="All Documents"
            ),
            gr.update(
                choices=list(documents.keys()),
                value=None
            ),
            project_overview()
        )


    # Upload callback

    upload_button.click(
        fn=upload_callback,
        inputs=[pdf_upload],
        outputs=[
            upload_status,
            library_output,
            document_scope,
            insight_scope,
            delete_dropdown,
            overview_output
        ]
    )


    # Delete callback

    delete_button.click(
        fn=delete_callback,
        inputs=[delete_dropdown],
        outputs=[
            delete_status,
            library_output,
            document_scope,
            insight_scope,
            delete_dropdown,
            overview_output
        ]
    )


    # AI chat callback

    send_button.click(
        fn=chat_with_ai,
        inputs=[
            question_input,
            chatbot,
            document_scope
        ],
        outputs=[
            chatbot,
            question_input
        ]
    )


    # Enter key callback

    question_input.submit(
        fn=chat_with_ai,
        inputs=[
            question_input,
            chatbot,
            document_scope
        ],
        outputs=[
            chatbot,
            question_input
        ]
    )


    # Clear chat callback

    clear_chat_button.click(
        fn=clear_chat,
        inputs=[],
        outputs=[chatbot]
    )


    # Research insights callback

    insight_button.click(
        fn=generate_insights,
        inputs=[insight_scope],
        outputs=[insight_output]
    )


    # Analytics callback

    refresh_btn.click(
        fn=refresh_analytics,
        inputs=[],
        outputs=[
            document_plot,
            keyword_plot,
            page_plot
        ]
    )


    # Saved conversations callback

    history_button.click(
        fn=get_saved_conversations,
        inputs=[],
        outputs=[history_output]
    )


    # Clear saved history callback

    clear_history_button.click(
        fn=clear_saved_conversations,
        inputs=[],
        outputs=[history_status]
    )


    # Overview callback

    demo.load(
        fn=project_overview,
        inputs=[],
        outputs=[overview_output]
    )


# ============================================================
# 20. LAUNCH
# ============================================================

print("Starting DocuMind AI...")

demo.launch(
    share=True,
    debug=True
)

Loading AI models... Please wait.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/tmp/ipykernel_2282/3741236506.py:879: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


AI models loaded successfully!
Starting DocuMind AI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9e08e5d658079d6ece.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9e08e5d658079d6ece.gradio.live
